In [1]:
import pandas as pd
import numpy as np
import os
os.environ ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
from langchain.embeddings import HuggingFaceInstructEmbeddings
from langchain.document_loaders import DirectoryLoader
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceHubEmbeddings,HuggingFaceInstructEmbeddings
from langchain.llms import HuggingFaceHub
from langchain.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains.question_answering import load_qa_chain
from langchain.chains import VectorDBQA
from langchain.llms import OpenAI
from langchain.llms.base import LLM
import time
import numpy as np
from typing import List
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

In [3]:
from typing import Any, Dict, Iterator, List, Mapping, Optional
from langchain_core.callbacks.manager import CallbackManagerForLLMRun
from langchain_core.language_models.llms import LLM
from langchain_core.outputs import GenerationChunk



In [4]:
from huggingface_hub import InferenceClient

/home/ramayana/madhan/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sentence_transformers.quantization import quantize_embeddings

In [6]:
class MyEmbeddings:
    def __init__(self):
        self.model =SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")

        
    # def get_embeddings(self, texts: List[str], model: str, prompt: str = None) -> np.ndarray:
    #     res = mxbai.embeddings(
    #         input=texts,
    #         model=model,
    #         prompt=prompt
    #     )
    #     embeddings = [entry.embedding for entry in res.data]
    #     return np.array(embeddings)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # return self.get_embeddings(
        #     texts,
        #     model_name,
        #     "Represent these documents for searching relevant sentences"
        # )
        return [self.model.encode(t).tolist() for t in texts]
    
    def embed_query(self, text:str) -> List[float]:
        return self.model.encode(text).tolist()
    
    

In [7]:
persist_directory=''
embedding=MyEmbeddings()

In [8]:
vectordb=Chroma(persist_directory=persist_directory, embedding_function=embedding)

In [29]:
# context_docs=vectordb.max_marginal_relevance_search("I Am A Government Job Aspirant, And In A Property Dispute, An Fir Under Sections 447, 341, 323, 354, 379, 504, 506, And 34 Of The Indian Penal Code Was Lodged Against My Father, My Uncle, And Me By My Aunt On May 30, 2019. Since My Date Of Birth Is January 2, 2002, My Age At The Time Of The Alleged Incident Was 17 Years, 4 Months, And 28 Days, Making Me A Juvenile. Due To A Lack Of Legal Awareness, I Was Unaware Of The Existence Of Any Juvenile Justice Act. The Concerned Court Granted Anticipatory Bail On July 3, 2019, And Since Then, We Have Been On Bail. It's Worth Noting That None Of The Offences Specified In These Sections Of The Indian Penal Code Carry Punishments Considered Heinous Under The Juvenile Justice (care And Protection Of Children) Act, 2015, As None Of The Punishments Exceeds Or Equals 7 Years Of Imprisonment. My Questions And Concerns: ● My Advocate Lied About My Age In Court And Portrayed Me As An Adult. He Also Told Me To Say That I Was Uneducated When Asked By The Judge. Luckily, The Judge Did Not Ask About My Education. I Don’t Know Why He Did It Or What Consequences It May Have For My Case Now. ● Despite Being Above The Age Of 16 At The Time Of The Alleged Incident, I Want My Case To Be Tried By The Juvenile Justice Board And Not A Children’s Court, As None Of The Offences Are Considered Heinous. How Can I Do It? How Can I Transfer My Case From A District Court, Where It Is Pending, To The Juvenile Justice Board? ● Since The Fir Was Filed By My Very Own Aunt, She Now Regrets Filing This False Fir In The Heat Of The Moment And Wants To Retract It. However, Since Section 354 Is Non-compoundable, It Must Be Contested. She Can Testify In Court That Nothing Of This Sort Has Happened, But Then Again, I Will Likely Be Acquitted Due To The Benefit Of The Doubt. So, Will Section 24 Of The Juvenile Justice (care And Protection Of Children) Act, 2015 Apply? Will My Records Be Deleted, And Will I Not Suffer From Any Sort Of Disqualification?")

In [30]:
# context=context_docs[0].page_content

In [31]:
max_tokens= 4000
client = InferenceClient(model="mistralai/Mixtral-8x7B-Instruct-v0.1", token="")

In [9]:
PROMPT='''
You are an experienced Indian legal consultant with expertise in various cases, legal sections, acts, and terminology. Users will approach you with their legal issues, and your role is to provide concise answers based on the given context. If you are unsure about the answer, simply respond with, "Sorry, I don't know."

Context: {context}

Question: {question}

'''

In [55]:
input_prompt=PROMPT.format(question="I Am A Government Job Aspirant, And In A Property Dispute, An Fir Under Sections 447, 341, 323, 354, 379, 504, 506, And 34 Of The Indian Penal Code Was Lodged Against My Father, My Uncle, And Me By My Aunt On May 30, 2019. Since My Date Of Birth Is January 2, 2002, My Age At The Time Of The Alleged Incident Was 17 Years, 4 Months, And 28 Days, Making Me A Juvenile. Due To A Lack Of Legal Awareness, I Was Unaware Of The Existence Of Any Juvenile Justice Act. The Concerned Court Granted Anticipatory Bail On July 3, 2019, And Since Then, We Have Been On Bail. It's Worth Noting That None Of The Offences Specified In These Sections Of The Indian Penal Code Carry Punishments Considered Heinous Under The Juvenile Justice (care And Protection Of Children) Act, 2015, As None Of The Punishments Exceeds Or Equals 7 Years Of Imprisonment. My Questions And Concerns: ● My Advocate Lied About My Age In Court And Portrayed Me As An Adult. He Also Told Me To Say That I Was Uneducated When Asked By The Judge. Luckily, The Judge Did Not Ask About My Education. I Don’t Know Why He Did It Or What Consequences It May Have For My Case Now. ● Despite Being Above The Age Of 16 At The Time Of The Alleged Incident, I Want My Case To Be Tried By The Juvenile Justice Board And Not A Children’s Court, As None Of The Offences Are Considered Heinous. How Can I Do It? How Can I Transfer My Case From A District Court, Where It Is Pending, To The Juvenile Justice Board? ● Since The Fir Was Filed By My Very Own Aunt, She Now Regrets Filing This False Fir In The Heat Of The Moment And Wants To Retract It. However, Since Section 354 Is Non-compoundable, It Must Be Contested. She Can Testify In Court That Nothing Of This Sort Has Happened, But Then Again, I Will Likely Be Acquitted Due To The Benefit Of The Doubt. So, Will Section 24 Of The Juvenile Justice (care And Protection Of Children) Act, 2015 Apply? Will My Records Be Deleted, And Will I Not Suffer From Any Sort Of Disqualification?",context=context)

In [56]:
print(client.text_generation(input_prompt,temperature=0.6,max_new_tokens=max_tokens))

Answer: Based on the given context, I can provide the following answers to your questions:

1. Your advocate's decision to misrepresent your age and portray you as an adult in court is concerning. This action could have serious consequences for your case, as it amounts to misrepresentation of facts before the court. It is essential to discuss this matter with your advocate and consider engaging a new one if necessary. You can inform the court about the correct facts and seek appropriate remedies.
2. To transfer your case from a district court to the Juvenile Justice Board, you can file a transfer application under Section 18 of the Juvenile Justice (Care and Protection of Children) Act, 2015. You need to provide evidence to establish that you were a juvenile at the time of the alleged incident. The court will then decide whether to transfer the case to the Juvenile Justice Board. Since none of the offences specified in the FIR are considered heinous, it is likely that the court will gr

## Mixtral Response Over test data

In [15]:
test_data=pd.read_csv('')

In [16]:
test_data.columns

Index(['title', 'question', 'answer', 'type'], dtype='object')

In [38]:
rows=[]

for i in tqdm(range(len(test_data))):

    question=test_data['question'][i]
    answer=test_data['answer'][i]
    type=test_data['type'][i]

    context_docs=vectordb.max_marginal_relevance_search(question.strip())
    context=''

    for k in range(2):
        context+=context_docs[k].page_content+'\n'
    
    input_prompt=PROMPT.format(question=question,context=context)
    
    for k in range(2):
        try:

            answer_generated = client.text_generation(input_prompt,temperature=0.6,max_new_tokens=max_tokens)
            rows.append([question,answer,answer_generated,context,type])
            break

        except Exception as e:
            print('Retrying=>',k+1)
            print(e)
            time.sleep(3)


100%|██████████| 100/100 [12:28<00:00,  7.48s/it]


In [40]:
result_df=pd.DataFrame(rows,columns=['question','answer','answer_generated','context','type'])

In [ ]:
result_df

In [43]:
result_df.to_csv('',index=False)

## LLama3-70b Response Over Test Data

In [ ]:
# pip install replicate

In [12]:
import replicate
os.environ['REPLICATE_API_TOKEN']=''

In [13]:
rows=[]

for i in tqdm(range(len(test_data))):

    question=test_data['question'][i]
    answer=test_data['answer'][i]
    type=test_data['type'][i]

    context_docs=vectordb.max_marginal_relevance_search(question.strip())
    context=''

    for k in range(2):
        context+=context_docs[k].page_content+'\n'
    
    input_prompt=PROMPT.format(question=question,context=context)
    
    for k in range(2):
        try:

            input={
                "top_k": 50,
                "top_p": 0.9,
                "prompt": input_prompt,
                "max_tokens": 1024,
                "min_tokens": 0,
                "temperature": 0.6,
                "prompt_template": "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
                "presence_penalty": 1.15,
                "frequency_penalty": 0.2
            }
            
            output=replicate.run(
                "meta/meta-llama-3-70b-instruct",
                input=input
            )
            answer_generated=""
            for item in output:
                answer_generated+=item+''
            rows.append([question,answer,answer_generated,context,type])

        except Exception as e:
            print('Retrying=>',k+1)
            print(e)
            time.sleep(3)


  3%|▎         | 3/100 [01:00<32:05, 19.86s/it]

Retrying=> 2
Server disconnected without sending a response.


 54%|█████▍    | 54/100 [21:52<17:08, 22.36s/it]

Retrying=> 2
Server disconnected without sending a response.


 55%|█████▌    | 55/100 [22:19<17:49, 23.76s/it]

Retrying=> 2
Server disconnected without sending a response.


100%|██████████| 100/100 [40:19<00:00, 24.19s/it]


In [17]:
result_df=pd.DataFrame(rows,columns=['question','answer','answer_generated','context','type'])

In [19]:
result_df=result_df.drop_duplicates(subset=['question'])

In [22]:
result_df=result_df.reset_index(drop=True)

In [24]:
result_df.to_csv('',index=False)

## LLama3-70b testing without context

In [42]:
PROMPT='''
You are an experienced Indian legal consultant with in-depth knowledge of legal cases, sections, acts, and terminology. Your task is to answer users' legal questions with clear and accurate information. If you don't know the answer to a question, respond with, "Sorry, I don't know."

Question: {question}

'''

In [14]:
rows=[]

for i in tqdm(range(len(test_data))):

    question=test_data['question'][i]
    answer=test_data['answer'][i]
    type=test_data['type'][i]
    
    input_prompt=PROMPT.format(question=question)
    
    for k in range(2):
        try:

            input={
                "top_k": 50,
                "top_p": 0.9,
                "prompt": input_prompt,
                "max_tokens": 1024,
                "min_tokens": 0,
                "temperature": 0.6,
                "prompt_template": "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
                "presence_penalty": 1.15,
                "frequency_penalty": 0.2
            }
            
            output=replicate.run(
                "meta/meta-llama-3-70b-instruct",
                input=input
            )
            answer_generated=""
            for item in output:
                answer_generated+=item+''
            rows.append([question,answer,answer_generated,type])

        except Exception as e:
            print('Retrying=>',k+1)
            print(e)
            time.sleep(3)


 84%|████████▍ | 84/100 [36:24<06:40, 25.04s/it]

Retrying=> 1
Server disconnected without sending a response.


100%|██████████| 100/100 [43:35<00:00, 26.15s/it]


In [15]:
result_df=pd.DataFrame(rows,columns=['question','answer','answer_generated','type'])

In [18]:
result_df=result_df.drop_duplicates(subset=['question']).reset_index(drop=True)

In [19]:
result_df

,question,answer,answer_generated,type
0,Dear Sir/mam Please Assist Me How And Where Sh...,"Dear Client, Anticipatory By Its Very Nature I...",A very specific and technical question!\n\nTo ...,Anticipatory_bail
1,Can A Person Get Bail After Being Sentenced By...,Dear Client If The Convicted Person Files An A...,"In India, the concept of bail after being sent...",Anticipatory_bail
2,My Sister Filed A False Fir Of Torture Against...,Dear Client The Surrender Slip You Received Af...,I understand your concern and the sensitive si...,Anticipatory_bail
3,My Relative Is In Judicial Custody In A False ...,"Dear Client, In India, The Granting Of Bail De...",I understand your concern and the situation yo...,Anticipatory_bail
4,Hi. I've Been Wrongly Accused For A 354 Case A...,"Dear Client, You Can Go Early To The Police St...","I understand your concern.\n\nFirstly, congrat...",Anticipatory_bail
...,...,...,...,...
95,My Friend Had Beaten A Sub Registrar Using His...,"Dear Sir, Yes, It Is Non Bailable If Section 3...",I'd be happy to help you with that!\n\nBased o...,Criminal
96,My Wife Has Filed Wrong Dvc Case 3 Years Back....,"Dear Client, Hon'ble Supreme Court Held That P...",I'd be happy to help you with your questions.\...,Criminal
97,Is It Illegal Or Crime To Use Someone Else Pro...,"Dear Sir, It Is Called Offence Of Impersonatio...","A very interesting question!\n\nIn India, usin...",Criminal
98,My First Motion Has Been Done And My Wife Had ...,"Dear Sir, You Can Lodge Contempt Of Court Case...",I understand your concern. \n\nIn this scenari...,Criminal


In [20]:
result_df.to_csv('',index=False)

## GPT Response with Mxbai

In [ ]:
# pip install openai

In [10]:
import openai


In [11]:
from openai import OpenAI
os.environ['OPENAI_API_KEY']=""


In [43]:
def ask_question(question,context=None):
    if context is None:
        response = openai.chat.completions.create(
          model="gpt-3.5-turbo",
          messages=[
            {
              "role": "system",
              "content": PROMPT.format(question=question)
            }
          ],
          temperature=0.6,
          max_tokens=1024
        )
    else:
        response = openai.chat.completions.create(
          model="gpt-3.5-turbo",
          messages=[
            {
              "role": "system",
              "content": PROMPT.format(question=question,context=context)
            }
          ],
          temperature=0.6,
          max_tokens=1024
        )
    
    return response.choices[0].message.content

In [44]:
rows=[]

for i in tqdm(range(len(test_data))):

    question=test_data['question'][i]
    answer=test_data['answer'][i]
    type=test_data['type'][i]

    # context_docs=vectordb.max_marginal_relevance_search(question.strip())
    # context=''

    # for k in range(2):
    #     context+=context_docs[k].page_content+'\n'
    
    input_prompt=PROMPT.format(question=question)
    
    for k in range(2):
        try:

            answer_generated=ask_question(question)
            rows.append([question,answer,answer_generated,type])

        except Exception as e:
            print('Retrying=>',k+1)
            print(e)
            time.sleep(3)


100%|██████████| 100/100 [13:12<00:00,  7.93s/it]


In [46]:
result_df=pd.DataFrame(rows,columns=['question','answer','answer_generated','type'])

In [48]:
result_df=result_df.drop_duplicates(subset=['question']).reset_index(drop=True)

In [50]:
result_df.to_csv('/home/ramayana/ayush/Legal_QA/New Output/openai_output.csv',index=False)